# COSMoS Observable Derivation

**Question.** If the unit of definition is the *observable* — the thing that actually receives a value — rather than the Biomedical Concept or the Dataset Specialization, how many observables does the current COSMoS package contain, how many does each BC hide, and how far are today's DSS boundaries from observable boundaries?

**Frame.** An observable is identified by a small set of axes, adopted from LOINC's structure rather than invented: *component* (what is measured), *system* (specimen or anatomical location), *scale* (quantitative / coded / text), and *method* only where it is pinned. Value type is not an axis: it follows from scale. Unit, TESTCD label and codelist binding are *recording* attributes, not identity. This notebook does **not** query LOINC; it uses only the LOINC codes the package itself pins on `--LOINC`.

**Scope.** Every DSS that carries a `--ORRES` variable, except the QRS domain (instrument-defined observations, where the axes fit badly). Events, Interventions and instrument rows are out of scope by construction.

**Companion.** `docs/Glucose_Siblings_BC_DSS_Proposal.html` — the worked glucose case this notebook generalises. The derivation rules below are the "Stage 0 rethought" described there in the 2026-08 section.

## Inputs

| File | Track | Role |
|---|---|---|
| `../cosmos-graph/interim/COSMoS_Graph.xlsx` | cosmos-graph | Graph projection of the COSMoS package (sheets `DSS`, `Variables`, `BC`) |

## Output

`reports/COSMoS_Observable_Derivation.xlsx`

| Sheet | Content |
|---|---|
| README | Provenance, derivation rules, summary counts |
| DSS_Coordinates | One row per in-scope DSS: derived axes + recording attributes |
| Observables | One row per distinct observable: member DSS, LOINC cardinality, unit sets |
| BC_Summary | One row per BC: DSS count, observable count, scale mix, split flag |
| Recording_Variants | Observables carried by more than one DSS — the DSS boundary is finer than the observable |

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('COSMoS OBSERVABLE DERIVATION')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

BASE_DIR = Path('..')                      # cosmos-bc-dss/
REPO_ROOT = BASE_DIR / '..'                # cdisc-for-ai/
GRAPH_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
REPORTS_DIR = BASE_DIR / 'reports'
REPORT_FILE = REPORTS_DIR / 'COSMoS_Observable_Derivation.xlsx'

assert GRAPH_FILE.exists(), f'Graph file not found: {GRAPH_FILE}'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

xl = pd.ExcelFile(GRAPH_FILE)
dss = xl.parse('DSS')
var = xl.parse('Variables')
bc = xl.parse('BC')
readme = xl.parse('ReadMe', header=None)
PACKAGE_DATE = str(dss['package_date'].dropna().max())[:10]     # latest DSS package date = the package release
GRAPH_GENERATED = str(readme.iloc[2, 0]).split(':', 1)[1].strip()
print(f'Package (latest DSS package_date): {PACKAGE_DATE}   graph generated: {GRAPH_GENERATED}')
print(f'DSS rows: {len(dss)}   Variables rows: {len(var)}   BC rows: {len(bc)}')

COSMoS OBSERVABLE DERIVATION
Run: 2026-08-29 11:02


Package (latest DSS package_date): 2026-07-14   graph generated: 2026-08-23
DSS rows: 1475   Variables rows: 13922   BC rows: 1475


## 2. Scope

In scope: DSS with a `--ORRES` variable, domain not QS.

In [2]:
var['suffix'] = var['variable_name'].str[2:]
has_orres = set(var.loc[var['suffix'] == 'ORRES', 'ds_id'])
scope = dss[dss['ds_id'].isin(has_orres) & (dss['domain'] != 'QS')].copy()
excluded_qs = int((dss['ds_id'].isin(has_orres) & (dss['domain'] == 'QS')).sum())
print(f'DSS with --ORRES : {len(has_orres)}')
print(f'  minus QS       : {excluded_qs}')
print(f'In scope         : {len(scope)}  across {scope["bc_id"].nunique()} BCs, {scope["domain"].nunique()} domains')
print(scope['domain'].value_counts().to_string())

DSS with --ORRES : 1264
  minus QS       : 17
In scope         : 1247  across 814 BCs, 20 domains
domain
IS    326
RS    157
LB    147
RE    135
RP     94
VS     78
SC     75
MK     50
GF     38
EG     33
FT     23
MB     14
DD     13
FA     12
SR     12
TU     11
UR     10
TR      9
MI      8
IE      2


## 3. Derivation rules

One value per DSS per axis. A variable is *pinned* when it carries `assigned_term_value` (one term); it is *open* when it offers a `value_list` (a choice, recorded per row) and *absent* when the variable is not in the DSS.

| Axis | Source | Rule |
|---|---|---|
| component | `bc_id` + `--TESTCD` assigned term + `--BDAGNT` / `--TSTDTL` / `--OBJ` assigned terms | The NCIt concept, refined by the test code and by the pinned target (allergen, organism, test detail) where one BC carries many |
| system | `--SPEC` assigned term, else `--LOC` assigned term | Pinned specimen or location. Open lists → `OPEN`; no variable → `NONE` |
| scale | `--ORRES` `data_type` and value binding; if `--ORRES` is untyped, fall back to `--STRESN` then `--STRESC` | `float`/`integer` → Quantitative · `text` with `value_list` or codelist → Coded · `text` otherwise → Text · `datetime` → Datetime |
| method | `--METHOD` assigned term | Pinned method only. Open list → `OPEN`; no variable → `NONE`; variable without pin or list → `UNBOUND` |

In the observable key the three non-pinned states (`OPEN`, `NONE`, `UNBOUND`) are collapsed to a single `UNPINNED` token: a difference between them says how the recording offers the axis, not what is observed. The per-DSS coordinate columns keep the distinct states.

Recording attributes (carried, not identity): unit set from `--ORRESU` (assigned or value_list), pinned LOINC codes from `--LOINC`, presence of a subject-state variable (`--FAST`, `--POS`), the `--ORRES` data type itself.

Ordinal vs nominal cannot be distinguished from the package (both are `text` with a bound value set), so they share the `Coded` scale. That is a known loss of resolution, stated rather than guessed.

In [3]:
def pick(df_ds, suffix):
    row = df_ds[df_ds['suffix'] == suffix]
    if row.empty:
        return 'NONE', None, None
    r = row.iloc[0]
    if pd.notna(r['assigned_term_value']):
        return 'PINNED', r['assigned_term_value'], r
    if pd.notna(r['value_list']):
        return 'OPEN', r['value_list'], r
    return 'UNBOUND', None, r


def scale_of(r):
    if r is None:
        return 'NONE'
    dt = r['data_type']
    if dt in ('float', 'integer'):
        return 'Quantitative'
    if dt == 'datetime':
        return 'Datetime'
    if dt == 'text':
        if pd.notna(r['value_list']) or pd.notna(r['codelist_submission_value']):
            return 'Coded'
        return 'Text'
    return 'UNTYPED'


rows = []
for ds_id, g in var[var['ds_id'].isin(scope['ds_id'])].groupby('ds_id', sort=False):
    st_testcd, testcd, _ = pick(g, 'TESTCD')
    targets = [t for t in (pick(g, s)[1] if pick(g, s)[0] == 'PINNED' else None for s in ('BDAGNT', 'TSTDTL', 'OBJ')) if t]
    component = testcd if not targets else testcd + ' / ' + ' / '.join(targets)
    st_spec, spec, _ = pick(g, 'SPEC')
    st_loc, loc, _ = pick(g, 'LOC')
    st_meth, meth, _ = pick(g, 'METHOD')
    st_unit, unit, _ = pick(g, 'ORRESU')
    st_loinc, loinc, _ = pick(g, 'LOINC')
    orres = g[g['suffix'] == 'ORRES'].iloc[0]
    scale_src = 'ORRES'
    if pd.isna(orres['data_type']):
        for fb in ('STRESN', 'STRESC'):
            fbrow = g[g['suffix'] == fb]
            if not fbrow.empty and pd.notna(fbrow.iloc[0]['data_type']):
                orres, scale_src = fbrow.iloc[0], fb
                break

    if st_spec == 'PINNED':
        system = spec
    elif st_loc == 'PINNED':
        system = loc
    elif st_spec == 'OPEN' or st_loc == 'OPEN':
        system = 'OPEN'
    else:
        system = 'NONE'

    method = meth if st_meth == 'PINNED' else st_meth
    loinc_codes = [] if loinc is None else [c.strip() for c in str(loinc).split(';') if c.strip()]

    rows.append({
        'ds_id': ds_id,
        'component_testcd': testcd,
        'component': component,
        'system': system,
        'system_source': 'SPEC' if st_spec == 'PINNED' else ('LOC' if st_loc == 'PINNED' else st_spec if st_spec != 'NONE' else st_loc),
        'scale': scale_of(orres),
        'scale_source': scale_src,
        'method': method,
        'orres_data_type': orres['data_type'],
        'orres_value_binding': 'value_list' if pd.notna(orres['value_list']) else ('codelist' if pd.notna(orres['codelist_submission_value']) else 'none'),
        'unit_set': unit,
        'unit_status': st_unit,
        'loinc_codes': ';'.join(loinc_codes) if loinc_codes else None,
        'loinc_n': len(loinc_codes),
        'has_state_var': ';'.join(sorted(s for s in g['suffix'] if s in ('FAST', 'POS'))) or None,
    })

coord = pd.DataFrame(rows).merge(scope[['ds_id', 'bc_id', 'domain', 'ds_short_name']], on='ds_id')
coord = coord.merge(bc[['bc_id', 'bc_short_name', 'result_scales']], on='bc_id', how='left')
coord['system_key'] = coord['system'].where(~coord['system'].isin(['OPEN', 'NONE']), 'UNPINNED')
coord['method_key'] = coord['method'].where(~coord['method'].isin(['OPEN', 'NONE', 'UNBOUND']), 'UNPINNED')
coord['observable_key'] = (coord['bc_id'] + '|' + coord['component'].astype(str) + '|'
                           + coord['system_key'].astype(str) + '|' + coord['scale'] + '|' + coord['method_key'].astype(str))
assert len(coord) == len(scope)
print(f'Coordinates derived for {len(coord)} DSS')
print()
print('scale         :', coord['scale'].value_counts().to_dict())
print('system status :', coord['system'].isin(['OPEN', 'NONE']).map({True: 'open/none', False: 'pinned'}).value_counts().to_dict())
print('method        :', coord['method'].isin(['OPEN', 'NONE', 'UNBOUND']).map({True: 'open/none', False: 'pinned'}).value_counts().to_dict())

Coordinates derived for 1247 DSS

scale         : {'Quantitative': 710, 'Coded': 374, 'Text': 157, 'Datetime': 6}
system status : {'open/none': 1003, 'pinned': 244}
method        : {'open/none': 1161, 'pinned': 86}


## 4. Observables

Collapse DSS on the observable key. A DSS set sharing one key differs only in recording attributes.

In [4]:
obs = (coord.groupby('observable_key')
       .agg(bc_id=('bc_id', 'first'), bc_short_name=('bc_short_name', 'first'),
            domain=('domain', lambda s: ';'.join(sorted(set(s)))),
            component=('component', 'first'), system=('system_key', 'first'),
            scale=('scale', 'first'), method=('method_key', 'first'),
            n_dss=('ds_id', 'nunique'), ds_ids=('ds_id', lambda s: ';'.join(sorted(s))),
            unit_sets=('unit_set', lambda s: ' | '.join(sorted(set(str(u) for u in s.dropna()))) or None),
            loinc_codes=('loinc_codes', lambda s: ';'.join(sorted(set(c for v in s.dropna() for c in v.split(';')))) or None))
       .reset_index())
obs['loinc_n'] = obs['loinc_codes'].fillna('').apply(lambda v: len([c for c in v.split(';') if c]))
obs['loinc_cardinality'] = pd.cut(obs['loinc_n'], [-1, 0, 1, 999], labels=['0', '1', '>1']).astype(str)

print(f'Distinct observables : {len(obs)}   from {len(coord)} DSS in {coord["bc_id"].nunique()} BCs')
print(f'Observables with >1 DSS (recording variants): {(obs["n_dss"] > 1).sum()}')
print()
print('LOINC cardinality per observable (pinned codes only):')
print(obs['loinc_cardinality'].value_counts().reindex(['0', '1', '>1']).to_string())

Distinct observables : 1187   from 1247 DSS in 814 BCs
Observables with >1 DSS (recording variants): 57

LOINC cardinality per observable (pinned codes only):
loinc_cardinality
0     1048
1       96
>1      43


## 5. BC summary — what each BC hides

`n_observables` > 1 means the BC name covers more than one observable. `n_scales` > 1 is the glucose seam: the same BC name over quantitative and coded results. `split_reason` names the axis that separates the observables.

In [5]:
def split_reason(g):
    reasons = []
    for ax, col in [('component', 'component'), ('system', 'system_key'), ('scale', 'scale'), ('method', 'method_key')]:
        if g[col].astype(str).nunique() > 1:
            reasons.append(ax)
    return ';'.join(reasons) or None


bcs = (coord.groupby('bc_id')
       .agg(bc_short_name=('bc_short_name', 'first'), result_scales=('result_scales', 'first'),
            domains=('domain', lambda s: ';'.join(sorted(set(s)))),
            n_dss=('ds_id', 'nunique'), n_observables=('observable_key', 'nunique'),
            n_scales=('scale', 'nunique'), scales=('scale', lambda s: ';'.join(sorted(set(s)))),
            n_systems=('system', 'nunique'), n_components=('component', 'nunique'))
       .reset_index())
bcs = bcs.merge(coord.groupby('bc_id').apply(split_reason).rename('split_reason').reset_index(), on='bc_id')
bcs['hides_observables'] = bcs['n_observables'] > 1

multi = bcs[bcs['n_dss'] > 1]
print(f'BCs in scope                    : {len(bcs)}')
print(f'BCs with >1 DSS                 : {len(multi)}')
print(f'  of which >1 observable        : {int(multi["hides_observables"].sum())}')
print(f'  of which >1 scale (the seam)  : {int((multi["n_scales"] > 1).sum())}')
print()
print('Split reason among BCs hiding observables:')
print(multi.loc[multi['hides_observables'], 'split_reason'].value_counts().to_string())

BCs in scope                    : 814
BCs with >1 DSS                 : 78
  of which >1 observable        : 56
  of which >1 scale (the seam)  : 24

Split reason among BCs hiding observables:
split_reason
system                           21
component                         9
component;scale                   9
system;scale                      4
scale                             4
component;system;scale;method     3
component;scale;method            2
system;scale;method               2
method                            2


/sessions/rcw-012c24cyu4hjqyay6tjcky3y/tmp/ipykernel_5/3274869254.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bcs = bcs.merge(coord.groupby('bc_id').apply(split_reason).rename('split_reason').reset_index(), on='bc_id')


## 6. Boundary comparison — DSS grain vs observable grain

Three relations between a DSS and its observable:

- **1:1** — the DSS *is* the observable.
- **DSS finer than observable** — several DSS share one key (recording variants: unit, TESTCD label, codelist binding only).
- **BC coarser than observable** — the BC covers several keys (the sibling case).

In [6]:
one_to_one = int((obs['n_dss'] == 1).sum())
finer = obs[obs['n_dss'] > 1]
print(f'Observables realised by exactly one DSS : {one_to_one}')
print(f'Observables realised by several DSS     : {len(finer)}  (covering {int(finer["n_dss"].sum())} DSS)')
print(f'BCs coarser than their observables      : {int(bcs["hides_observables"].sum())}')
print()
print('Recording variants — largest groups:')
print(finer.sort_values('n_dss', ascending=False)[['bc_short_name', 'system', 'scale', 'n_dss', 'ds_ids']].head(12).to_string(index=False))

Observables realised by exactly one DSS : 1130
Observables realised by several DSS     : 57  (covering 117 DSS)
BCs coarser than their observables      : 56

Recording variants — largest groups:
                             bc_short_name   system        scale  n_dss                                                      ds_ids
                              Tumor Status UNPINNED        Coded      3 NEW_TUMSTATE_RECIST1_1;NONNODAL_TUMSTATE_RECIST1_1;TUMSTATE
                          Overall Response UNPINNED        Coded      3                      OVRLRESP;OVRLRESP_LUGANO;OVRLRESP_RANO
                 Response in Target Lesion UNPINNED        Coded      3                         TRGRESP;TRGRESP_LUGANO;TRGRESP_RANO
         Confluent Tumor Masses Assessment UNPINNED        Coded      2                                   TUMERGE;TUMERGE_RECIST1_1
Microbial-induced IgM Antibody Measurement UNPINNED Quantitative      2                                   MBIMABNGLI4;MBIMABNGLI4PN
Microbial-ind

## 7. Glucose check

The worked case, at observable grain. Expect more observables than the proposal's four siblings: the observable is at LOINC grain (system is an axis), while a *sibling BC* in the proposal is a group of observables sharing one scale and one interpretation regime. Note also what the package says about `GLUCSERPL`.

In [7]:
gl = coord[coord['bc_id'] == 'C105585'].sort_values('observable_key')
print(gl[['ds_id', 'system', 'scale', 'method', 'orres_data_type', 'unit_set', 'loinc_codes']].to_string(index=False))
print()
print(f'Glucose observables: {gl["observable_key"].nunique()}')

       ds_id             system        scale     method orres_data_type            unit_set      loinc_codes
      GLUCPE INTERSTITIAL FLUID Quantitative    UNBOUND           float    mg/dL;g/L;mmol/L 99504-3;105272-9
     GLUCBLD              BLOOD Quantitative    UNBOUND           float        mg/dL;mmol/L   2339-0;15074-8
      GLUCPL             PLASMA Quantitative       NONE           float    mg/dL;g/L;mmol/L             None
     GLUCSER              SERUM Quantitative       NONE           float    mg/dL;g/L;mmol/L             None
   GLUCSERPL              SERUM Quantitative       NONE           float    mg/dL;g/L;mmol/L   14749-6;2345-7
    GLUCURIN              URINE Quantitative       NONE           float mg/dL;mmol/L;umol/L   15076-3;2350-7
      GLUCUA              URINE         Text TEST STRIP            text                None          25428-4
GLUCURINPRES              URINE         Text       NONE            text                None           2349-9

Glucose observable

## 8. Report

In [8]:
YELLOW, YELLOW_DATA = 'FFD700', 'FFFCE8'
GREY, GREY_DATA = '7F7F7F', 'F2F2F2'
THIN = Side(style='thin', color='999999')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

DERIVED_COLS = {'component', 'system', 'system_source', 'scale', 'scale_source', 'method', 'observable_key', 'n_observables', 'n_scales', 'scales',
                'n_systems', 'n_components', 'split_reason', 'hides_observables', 'loinc_n', 'loinc_cardinality', 'n_dss'}


def write_sheet(wb, name, df, wide=()):
    ws = wb.create_sheet(name)
    ws.append(list(df.columns))
    for c, col in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=c)
        fill = GREY if col in DERIVED_COLS else YELLOW
        cell.fill = PatternFill('solid', fgColor=fill)
        cell.font = Font(bold=True, color='FFFFFF')
        cell.border = BORDER
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    for r in df.itertuples(index=False):
        ws.append([None if (isinstance(v, float) and pd.isna(v)) else v for v in r])
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            col = df.columns[cell.column - 1]
            cell.fill = PatternFill('solid', fgColor=GREY_DATA if col in DERIVED_COLS else YELLOW_DATA)
            cell.border = BORDER
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    for c, col in enumerate(df.columns, 1):
        width = max([len(str(col))] + [len(str(v)) for v in df[col].head(300)])
        ws.column_dimensions[get_column_letter(c)].width = min(width + 2, 90 if col in wide else 40)
    ws.freeze_panes = 'A2'
    return ws


readme_lines = [
    'COSMoS Observable Derivation',
    '',
    f'Generated: {datetime.now():%Y-%m-%d %H:%M}',
    f'Source: cosmos-graph/interim/COSMoS_Graph.xlsx (COSMoS package {PACKAGE_DATE})',
    'Notebook: cosmos-bc-dss/notebooks/COSMoS_Observable_Derivation.ipynb',
    '',
    'QUESTION',
    'If the unit of definition is the observable (the thing that receives a value) rather than the BC or the DSS, how many observables does the package contain, how many does each BC hide, and how far are DSS boundaries from observable boundaries?',
    '',
    'SCOPE',
    f'DSS carrying a --ORRES variable, excluding QS: {len(coord)} DSS, {coord["bc_id"].nunique()} BCs, {coord["domain"].nunique()} domains.',
    '',
    'DERIVATION RULES',
    'component = bc_id + --TESTCD assigned term + pinned --BDAGNT / --TSTDTL / --OBJ terms.',
    'system = --SPEC assigned term, else --LOC assigned term; open value list -> OPEN; no variable -> NONE.',
    'scale = from --ORRES (fallback --STRESN, --STRESC when --ORRES is untyped): float/integer -> Quantitative; text with value_list or codelist -> Coded; text otherwise -> Text; datetime -> Datetime. Ordinal vs nominal is not distinguishable from the package and is not guessed.',
    'method = --METHOD assigned term only; open list -> OPEN; no variable -> NONE; variable without pin or list -> UNBOUND.',
    'observable_key = bc_id | component | system | scale | method, with the non-pinned states (OPEN / NONE / UNBOUND) collapsed to UNPINNED: a difference between them is a recording shape, not an identity.',
    'Recording attributes (not identity): unit set, --ORRES data type, LOINC codes as pinned on --LOINC, presence of --FAST / --POS.',
    'No external LOINC lookup. LOINC cardinality counts pinned codes only.',
    '',
    'SUMMARY',
    f'Distinct observables: {len(obs)}',
    f'Observables realised by one DSS: {one_to_one}; by several DSS (recording variants): {len(finer)}',
    f'BCs with >1 DSS: {len(multi)}; of which hiding >1 observable: {int(multi["hides_observables"].sum())}; of which mixing scales: {int((multi["n_scales"] > 1).sum())}',
    'LOINC cardinality per observable: ' + ', '.join(f'{k}: {v}' for k, v in obs['loinc_cardinality'].value_counts().reindex(['0', '1', '>1']).items()),
    f'LOINC pins: {int((coord["loinc_n"] > 0).sum())} of {len(coord)} in-scope DSS carry pinned codes - the LOINC cross-check covers that subset only.',
    '',
    'SHEETS',
    'DSS_Coordinates - one row per in-scope DSS with derived axes and recording attributes.',
    'Observables - one row per distinct observable key.',
    'BC_Summary - one row per BC; hides_observables and split_reason name the seam.',
    'Recording_Variants - observables carried by more than one DSS.',
    '',
    'COLOR CONVENTION',
    'Yellow FFD700 header = value taken from the COSMoS package (graph projection). Grey 7F7F7F header = derived in this notebook.',
]

wb = Workbook()
ws = wb.active
ws.title = 'README'
ws.column_dimensions['A'].width = 80
ws['A1'] = readme_lines[0]
ws['A1'].fill = PatternFill('solid', fgColor='595959')
ws['A1'].font = Font(bold=True, color='FFFFFF')
for i, line in enumerate(readme_lines[1:], start=2):
    ws.cell(row=i, column=1, value=line).alignment = Alignment(wrap_text=True, vertical='top')

coord_out = coord[['ds_id', 'bc_id', 'bc_short_name', 'domain', 'ds_short_name', 'result_scales', 'component_testcd', 'component',
                   'system', 'system_source', 'scale', 'scale_source', 'method', 'orres_data_type', 'orres_value_binding',
                   'unit_set', 'unit_status', 'loinc_codes', 'loinc_n', 'has_state_var', 'observable_key']]
write_sheet(wb, 'DSS_Coordinates', coord_out, wide=('observable_key', 'unit_set'))
write_sheet(wb, 'Observables', obs[['observable_key', 'bc_id', 'bc_short_name', 'domain', 'component', 'system', 'scale',
                                    'method', 'n_dss', 'ds_ids', 'unit_sets', 'loinc_codes', 'loinc_n', 'loinc_cardinality']],
            wide=('observable_key', 'ds_ids', 'unit_sets'))
write_sheet(wb, 'BC_Summary', bcs.sort_values(['hides_observables', 'n_observables'], ascending=[False, False]))
write_sheet(wb, 'Recording_Variants', finer.sort_values('n_dss', ascending=False)[['observable_key', 'bc_short_name', 'domain', 'system',
                                                                                    'scale', 'method', 'n_dss', 'ds_ids', 'unit_sets']],
            wide=('observable_key', 'ds_ids', 'unit_sets'))
wb.save(REPORT_FILE)
print(f'Written: {REPORT_FILE}')

Written: ../reports/COSMoS_Observable_Derivation.xlsx
